<a href="https://colab.research.google.com/github/gyeongee/TIL/blob/main/%EC%BD%98%ED%85%90%EC%B8%A0_%EA%B8%B0%EB%B0%98_%ED%95%84%ED%84%B0%EB%A7%81_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

콘텐츠 기반 필터링은 사용자가 특정 영화를 감상하고 그 영화를 좋아했다면 그 영화와 비슷한 특성/속성,구성 요소 등을 가진 다른 영화를 추천하는 것입니다.

In [40]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [41]:
df = pd.read_csv('/content/tmdb_5000_movies.csv')
df.head(3)

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

homepage,overview,runtime,tagline 피처에 결측치가 있고 homepage 변수는 결측치가 너무 많으므로 삭제하는 것이 좋을 것 같습니다.

In [43]:
# 사용할 피처 추출
df=df[['id','title','genres','vote_count','vote_average','popularity','keywords','overview']]


In [44]:
df.columns

Index(['id', 'title', 'genres', 'vote_count', 'vote_average', 'popularity',
       'keywords', 'overview'],
      dtype='object')

'genres','keywords'등과 같이 파이썬 리스트(list) 내부에 여러 개의 딕셔너리가 있는 형태의 문자열로 표기돼 있습니다.

'genres' 칼럼의 문자열을 분해해서 개별 장르를 파이썬 리스트 객체로 추출하겠습니다.

파이썬 ast 모듈의 l`iteral_eval()` 함수를 이용하면 이 문자열을 문자열이 의미하는 `list[dict1,dict2]`객체로 만들 수 있습니다.

In [45]:
from ast import literal_eval
df['genres']=df['genres'].apply(literal_eval)
df['keywords']=df['keywords'].apply(literal_eval)

이제 genres 칼럼은 문자열이 아니라 실제 리스트 내부에 여러 장르 딕셔너리로 구성된 객체를 가집니다.

genres 칼럼에서 'name'에 해당하는 값을 추출하겠습니다.

In [46]:
df['genres']=df['genres'].apply(lambda x:[y['name'] for y in x])
df['keywords']=df['keywords'].apply(lambda x:[y['name']for y in x])

df[['genres','keywords']]

,genres,keywords
0,"[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon..."
1,"[Adventure, Fantasy, Action]","[ocean, drug abuse, exotic island, east india ..."
2,"[Action, Adventure, Crime]","[spy, based on novel, secret agent, sequel, mi..."
3,"[Action, Crime, Drama, Thriller]","[dc comics, crime fighter, terrorist, secret i..."
4,"[Action, Adventure, Science Fiction]","[based on novel, mars, medallion, space travel..."
...,...,...
4798,"[Action, Crime, Thriller]","[united states–mexico barrier, legs, arms, pap..."
4799,"[Comedy, Romance]",[]
4800,"[Comedy, Drama, Romance, TV Movie]","[date, love at first sight, narration, investi..."
4801,[],[]


### 장르 콘텐츠 유사도 측정

여러 개의 개별 장르 리스트로 구성돼 있는 `genres`컬럼을 문자열로 변경한 뒤 이를 CountVectorizer로 피처 벡터화한 행렬 데이터 값을 코사인 유사도와 비교하겠습니다.

In [47]:
from sklearn.feature_extraction.text import CountVectorizer

# CountVectorizer를 적용하기 위해 공백문자로 word 단위가 구분되는 문자열로 변환
df['genres_literal']=df['genres'].apply(lambda x:(' ').join(x))
count_vect=CountVectorizer(min_df=0.0,ngram_range=(1,2))
genre_mat=count_vect.fit_transform(df['genres_literal'])
print(genre_mat.shape)

(4803, 276)


CountVectorizer 로 변환해 4803개의 레코드와 276개의 개별 단어 피처로 벡터 행렬이 만들어졌습니다.

이렇게 생성된 피처 벡터 행렬에 사이킷런의 `cosine_similarity()`를 이용해 코사인 유사도를 계산하겠습니다.

In [48]:
# 데이터의 행별 장르 유사도 값
from sklearn.metrics.pairwise import cosine_similarity

genres_sim = cosine_similarity(genre_mat,genre_mat)
print(genres_sim.shape)
print(genres_sim[:2])

(4803, 4803)
[[1.         0.59628479 0.4472136  ... 0.         0.         0.        ]
 [0.59628479 1.         0.4        ... 0.         0.         0.        ]]


In [49]:
# genres_sim 객체의 기준 행별로
#비교 대상이 되는 행의 유사도 값이 높은 순으로 정렬된 행령의 위치 인덱스 값 추출
genre_sim_sorted_ind = genres_sim.argsort()[:,::-1]
print(genre_sim_sorted_ind[:1])

[[   0   46 3494 ... 3331 3333 2031]]


0번 레코드의 경우 자신인 0번 레코드를 제외하면 3494번 레코드가 가장 유사도가 높고, 가장 유사도가 낮은 레코드는 2031 입니다.

### 장르 콘텐츠 필터링을 이용한 영화추천

In [50]:
# 장르 유사도에 따라 영화 추천하는 함수
# find_sim_moive() 생성
def find_sim_movie(df,sorted_ind,title_name,top_n=10):
  # 인자로 입력된 df의 DataFrame에서 'title'칼럼이 입력된 title_name 값인 DatFrame 추출
  title_movie=df[df['title']==title_name]
  # sorted_ind 인자로 입력된 genre_sim_sorted_ind 객체에서 유사도 순으로 top_n개의 index 추출
  title_index=title_movie.index.values
  similar_indexes=sorted_ind[title_index,:(top_n)]

  #추출된 top_n index 출력 top_n index는 2차원 데이터임
  # dataframe에서 index로 사용하기 위해서 1차원 array로 변경
  print(similar_indexes)
  similar_indexes = similar_indexes.reshape(-1)

  return df.iloc[similar_indexes]

In [51]:
# 영화 '대부'와 장르별로 유사한 영화 10개 추천
similar_movies=find_sim_movie(df,genre_sim_sorted_ind,'The Godfather',10)
similar_movies[['title','vote_average']]

[[1881 3378 3866 1370 1464  588 3887 3594 2839  892]]


,title,vote_average
1881,The Shawshank Redemption,8.5
3378,Auto Focus,6.1
3866,City of God,8.1
1370,21,6.5
1464,Black Water Transit,0.0
588,Wall Street: Money Never Sleeps,5.8
3887,Trainspotting,7.8
3594,Spring Breakers,5.0
2839,Rounders,6.9
892,Casino,7.8


개선이 필요한 결과가 나왔습니다.

이번에는 좀 더 많은 후보군을 선정한 뒤 영화의 평점에 따라 필터링하여 최종 추천 하는 방식으로 변경하겠습니다.

In [52]:
df[['title','vote_average','vote_count']].sort_values('vote_average',ascending=False)[:10]

,title,vote_average,vote_count
4662,Little Big Top,10.0,1
3519,Stiff Upper Lips,10.0,1
4045,"Dancer, Texas Pop. 81",10.0,1
4247,Me You and Five Bucks,10.0,2
3992,Sardaarji,9.5,2
2386,One Man's Hero,9.3,2
1881,The Shawshank Redemption,8.5,8205
2970,There Goes My Baby,8.5,2
3337,The Godfather,8.4,5893
2796,The Prisoner of Zenda,8.4,11


왜곡된 평점을 회피할 수 있도록 평점에 평가 횟수를 반영할 수 있는 새로운 평가 방식인 유명한 영화 평점 사이트인 IMDB에서 사용하는 가중치가 부여된 평점(Weighted Rating) 방식을 사용하겠습니다.

가중평점 = (v/(v+m))*R+(m/(v+m))*C

각 변수의 의미는 다음과 같습니다.
- v:개별 영화에 평점을 투표한 횟수(vote_count)
- m:평점을 부여하기 위한 최소 투표 횟수
- R:개별 영화에 대한 평균 평점(vote_average)
- C:전체 영화에 대한 평균 평점

In [53]:
# m값은 전체 투표 횟수에서 상위 60%에 해당하는 횟수 기준
C=df['vote_average'].mean()
m=df['vote_count'].quantile(0.6)
print(f'C:{round(C,3)},m:{round(m,3)}')

C:6.092,m:370.2


In [54]:
# 기존 평점을 새로운 가중 평점으로 변경하는 함수를 생성하고
#이를 이용해 새로운 평점 정보인 vote_weighted 값을 만듦
percentile=0.6
m = df['vote_count'].quantile(percentile)
C=df['vote_average'].mean()

def weighted_vote_average(record):
  v=record['vote_count']
  R=record['vote_average']

  return ((v/(v+m)) * R) + ( (m/(m+v)) * C)

df['weighted_vote']=df.apply(weighted_vote_average,axis=1)


In [55]:
# 사용할 피처 추출
df=df[['title','vote_count','vote_average','weighted_vote']].sort_values('weighted_vote',ascending=False)[:10]
df.head()

,title,vote_count,vote_average,weighted_vote
1881,The Shawshank Redemption,8205,8.5,8.396052
3337,The Godfather,5893,8.4,8.263591
662,Fight Club,9413,8.3,8.216455
3232,Pulp Fiction,8428,8.3,8.207102
65,The Dark Knight,12002,8.2,8.136930


To

In [59]:
def find_sim_movie(df,sorted_ind,title_name,top_n=10):
  # 인자로 입력된 df의 DataFrame에서 'title'칼럼이 입력된 title_name 값인 DatFrame 추출
  title_movie=df[df['title']==title_name]
  title_index=title_movie.index.values
  # top_n의 2배에 해당하는 장르 유사성이 높은 인덱스 추출
  similar_indexes=sorted_ind[title_index,:(top_n*2)]
  similar_indexes=similar_indexes.reshape(-1)
  # 기존 영화 인덱스는 제외
  similar_indexes=similar_indexes[similar_indexes != title_index]

  #top_n의 2배에 해당하는 후보군에서 weighted_vote가 높은 순으로 top_n만큼 추출
  return df.iloc[similar_indexes].sort_values('weighted_vote',ascending=False)[:top_n]

similar_movies=find_sim_movie(df,genre_sim_sorted_ind,'The GodFather',10)
similar_movies[['title','vote_average','weighted_vote']]

,title,vote_average,weighted_vote
